In [29]:
import joblib 
from transformers import AutoTokenizer, AutoModelForSequenceClassification 
import torch

In [4]:
def load_model(model_type, model_name):
    '''
    model_type : 'ml' or 'transformer' 
    model_name : example - 'linear_svc', 'logistic_regression', 'naive_bayes', 'distilbert'
    '''
    if model_type == 'ml':
        model_path = f"../models/ml/{model_name}.joblib"
        model = joblib.load(model_path)
        return model, None 
    
    elif model_type == 'transformer':
        model_path = f"../models/transformers/{model_name}"
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        model.eval()
        return model, tokenizer
    
    else:
        raise ValueError("model type must be 'ml' or 'transformer'")

In [21]:
LABEL_MAP = {
    0: 'Negative',
    1: 'Positive'
}

In [42]:
def predict(text, model_type, model_name, LABEL_MAP):
    model, tokenizer = load_model(model_type, model_name)

    if model_type == "ml":
        pred = model.predict([text])[0]
        return LABEL_MAP[pred]
        
    elif model_type == "transformer":
        inputs = tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            pred = torch.argmax(probs, dim=-1).item()
        return LABEL_MAP[pred]

In [44]:
text = "Movie is great"
model_type = "ml"
model_name = "naive_bayes"
predict(text, model_type, model_name, LABEL_MAP)

'Positive'

In [45]:
text = "Movie is great"
model_type = "transformer"
model_name = "distilbert"
predict(text, model_type, model_name, LABEL_MAP)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2710.16it/s, Materializing param=pre_classifier.weight]                                  


'Positive'